# 第 10 章　模块与程序包

> **本章定位**：从"一个文件写完"到"按职责拆分、可复用、可分发的工程结构"。
>
> **知识点范围**：模块的概念与作用、`import 模块` / `from 模块 import 名称` / `import ... as 别名`、`if __name__ == "__main__"`、标准库 `random` 与 `datetime`、自定义模块的编写、程序包目录与 `__init__.py`、相对导入。
>
> **本章约定**：可用前 9 章全部内容；只使用标准库，**不安装第三方库**（第三方库放在第 13 章结合具体需求讲解）。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 10.1 — 10.5 | 导入机制、模块与包 + 随堂示例 |
| 第二部分 | 代码示范 | 案例 10-1、案例 10-2 | 自定义模块 / 标准库仿真 |
| 第三部分 | 布置练习 | 练习 10-1、练习 10-2 | 模块化统计 / 搭建程序包 |
| 第四部分 | 习题答案 | 答案 10-1、答案 10-2 | 对照解析与评分要点 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 10.1 | 模块概念与三种导入形式 | 10.2 | 案例 10-1 | 练习 10-1 |
| 10.2 | 自定义模块与 `__doc__` | 10.3 | 案例 10-1 | 练习 10-1 |
| 10.3 | `if __name__ == "__main__"` | 10.4 | 案例 10-1（可加） | 练习 10-1 |
| 10.4 | 标准库 `random` 与 `datetime` | 10.5 | 案例 10-2 | —— |
| 10.5 | 程序包、`__init__.py` 与相对导入 | 10.6 | —— | 练习 10-2 |

## 教学提示（教师用）

- **建议课时**：4 学时（2 次课），对应课程标准模块 10。
- **讲解主线**：模块解决"代码可复用、命名不冲突"，包解决"多文件如何组织"，`if __name__ == "__main__"` 解决"被导入时不误执行"。三种导入形式要现场演示差异。
- **学生常见卡点**：`import 模块` 后忘记写模块名前缀；`from ... import *` 污染命名空间；包内程序必须在上一级目录用 `python -m 包.模块` 运行；相对导入只在包内有效；自定义模块名与标准库同名（如 `random.py`）导致导入错乱。
- **课堂活动建议**：把第 6 章的函数库直接另存为 `traffic_utils.py`，只加 `import` 就能复用，让学生体会"零改动复用"的价值。

---

# 第一部分　知识讲解

## 10.1 场景导入：300 行的文件为什么必须拆

当一个文件超过 300 行，"拥堵判定"和"文件读写"混在一起，改一处就要上下翻找。工程化的做法是**按职责拆成多个 `.py` 文件**：计算归计算、数据归数据、界面归界面。

拆分的直接收益有三条：

1. **复用**：计算函数写一次，多个程序都能 `import`；
2. **定位**：出问题只需看对应模块，不必通读全文；
3. **协作**：两人分别改 `calc.py` 与 `main.py`，互不冲突。

再往上，把若干相关模块放进一个目录并加上 `__init__.py`，就是**程序包**——`pip install` 装下来的东西正是这个形态。

## 10.2 知识点 10.1　模块概念与三种导入形式

### 10.2.1 核心概念

**模块**就是一个 `.py` 文件。文件名（去掉 `.py`）即模块名，模块内的函数、变量、类都是它的"成员"。

| 导入写法 | 调用方式 | 适用场景 | 命名空间 |
|---|---|---|---|
| `import traffic_utils` | `traffic_utils.capacity(3)` | 模块名短，或需明确来源 | 独立 |
| `import traffic_utils as tu` | `tu.capacity(3)` | 模块名长，**本章推荐** | 独立 |
| `from traffic_utils import capacity` | `capacity(3)` | 只用一两个函数 | 并入当前 |
| `from traffic_utils import *` | `capacity(3)` | **不推荐**，易与本地变量重名 | 并入当前 |

### 10.2.2 语法要点

- 导入语句习惯放在文件**顶部**，标准库在前、第三方库居中、自定义模块在后；
- 模块只在**首次**导入时执行一次，再次 `import` 不会重复执行；
- 导入失败抛 `ModuleNotFoundError`，最常见原因是"模块不在当前目录且不在 `sys.path` 中"；
- `dir(模块)` 查看成员，`模块.__doc__` 查看模块文档字符串。

### 10.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 模块与主程序不在同一目录 | `ModuleNotFoundError` | 放同目录，或在上级目录运行 |
| 文件名与标准库同名 | 导入到错误模块 | 如 `random.py` 会遮蔽标准库 |
| `import traffic-utils` | `SyntaxError` | 模块名不能含连字符 |
| 用了 `from ... import *` 后函数重名 | 调用到意外版本 | 改用别名导入 |

### 10.2.4 随堂示例：三种导入形式与调用写法

**核心要点**：验证三种导入写法的差别：`import 模块` 要加模块名前缀、`from 模块 import 名称` 可直接用、`import 模块 as 别名` 可简化长名字。

In [ ]:
import math                        # 形式一：导入整个模块
from math import sqrt              # 形式二：只导入一个函数
import random as rd                # 形式三：导入并起别名

print("math.ceil(3.2)：", math.ceil(3.2))
print("sqrt(16)：", sqrt(16))       # 不需要写 math.

rd.seed(2026)                      # 固定种子，结果可复现
print("rd.randint(1, 10)：", rd.randint(1, 10))

**预期输出**：

```text
math.ceil(3.2)： 4
sqrt(16)： 4.0
rd.randint(1, 10)： 2
```

## 10.3 知识点 10.2　自定义模块与 `__doc__`

### 10.3.1 核心概念

自定义模块的编写规范与函数类似，只是"层级更高"：

| 位置 | 内容 | 作用 |
|---|---|---|
| 文件开头 | 三引号文档字符串 | 说明模块用途、作者、版本 |
| 文档字符串之后 | 模块级常量 | 如 `DEFAULT_CAPACITY = 2200` |
| 常量之后 | 函数 / 类定义 | 对外提供的功能 |
| 末尾 | `if __name__ == "__main__":` 自测 | 直接运行时才执行 |

### 10.3.2 语法要点

- 模块文档字符串必须是**文件第一条语句**，否则 `__doc__` 为 `None`；
- `模块.__doc__` 返回整段文档，取首行常用 `.__doc__.strip().splitlines()[0]`；
- 模块内可定义"私有"成员（`_name` 前缀），`from ... import *` 不会导入它们。

### 10.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 文档字符串前有注释或代码 | `__doc__` 为 `None` | 注释可放在文档字符串内或其后 |
| 模块名与变量名冲突 | 调用异常 | 命名区分 |
| 修改模块后未重启内核 | 仍是旧版本 | 重启内核或 `importlib.reload` |

### 10.3.4 随堂示例：用 __doc__ 读取模块说明

**核心要点**：验证模块开头的三引号字符串就是模块文档，可用 `模块.__doc__` 读取；自定义模块与标准库遵循完全相同的规则（案例 10-1 会新建 `traffic_utils.py` 验证）。

In [ ]:
import math

print("模块全名：", math.__name__)
print("文档首行：", math.__doc__.strip().splitlines()[0])
print("模块内常量 pi：", round(math.pi, 4))

# dir() 会把 __doc__ 这类内部名称一并列出，需过滤掉下划线开头的
public = [n for n in dir(math) if not n.startswith("_")]
print("按字母序的前 6 个公开名称：", public[:6])

**预期输出**：

```text
模块全名： math
文档首行： This module provides access to the mathematical functions
模块内常量 pi： 3.1416
按字母序的前 6 个公开名称： ['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2']
```

## 10.4 知识点 10.3　`if __name__ == "__main__"`

### 10.4.1 核心概念

每个模块都有一个内置变量 `__name__`：

- **被直接运行**时，`__name__` 等于 `"__main__"`；
- **被导入**时，`__name__` 等于模块名（如 `"flow_stats"`）。

因此把自测代码放进 `if __name__ == "__main__":` 块，就能实现"直接运行时自测、被导入时不打扰"。

### 10.4.2 语法要点

| 场景 | `__name__` 的值 | 自测块是否执行 |
|---|---|---|
| `python flow_stats.py` | `"__main__"` | 执行 |
| `import flow_stats` | `"flow_stats"` | 不执行 |
| `python -m traffic_pkg.main` | `"__main__"` | 执行 |

### 10.4.3 常见用法与易错点

- 自测代码**不要**写成 `if __name__ == "main":`（少下划线）或 `== __main__`（少了引号）。
- 把"演示数据"写进模块顶层而不加保护，导入时就会打印一堆测试输出，污染调用方界面。
- 主程序（入口文件）也应加上该保护，便于被其他程序当作模块导入。

### 10.4.4 随堂示例：看 __name__ 判断运行方式

**核心要点**：验证 `__name__` 的取值规律：直接运行（含 Notebook 单元格）时为 `\「__main__\」`，被其他文件导入时才是模块名。这就是「自测代码只在直接运行时执行」的原理。

In [ ]:
print("当前 __name__ 的值：", __name__)

if __name__ == "__main__":
    print("→ 直接运行：以下自测代码会执行")
else:
    print("→ 被导入：以下自测代码不会执行")

print("—— 自测开始 ——")
print("通行能力 3 车道：", 3 * 2200, "辆/h")

**预期输出**：

```text
当前 __name__ 的值： __main__
→ 直接运行：以下自测代码会执行
—— 自测开始 ——
通行能力 3 车道： 6600 辆/h
```

## 10.5 知识点 10.4　标准库 `random` 与 `datetime`

### 10.5.1 核心概念

标准库随 Python 安装，`import` 即可用，无需 `pip`。本章用两个：

| 模块 | 常用成员 | 用途 |
|---|---|---|
| `random` | `seed()` `randint()` `uniform()` `choice()` `random()` | 生成模拟车流数据 |
| `datetime` | `datetime.now()` `date.today()` `timedelta()` `strftime()` | 时间戳、日期加减、格式化 |

### 10.5.2 语法要点

| 函数 | 返回 | 示例 |
|---|---|---|
| `random.seed(n)` | `None` | 固定随机序列，**批改作业必加** |
| `random.randint(a, b)` | 整数，含两端 | `randint(600, 1800)` |
| `random.uniform(a, b)` | 浮点，含两端近似 | `uniform(15, 85)` |
| `random.choice(seq)` | 序列中一个元素 | `choice(["京港澳", "京开"])` |
| `random.random()` | `0 ≤ x < 1` 的浮点 | 概率抽样 |
| `datetime.datetime.now()` | datetime 对象 | 当前时间 |
| `obj.strftime("%Y-%m-%d %H:%M:%S")` | 字符串 | 时间格式化 |
| `date + timedelta(days=1)` | 新日期 | 明天/昨天 |

**`strftime` 常用占位符**：`%Y` 四位年、`%m` 两位月、`%d` 两位日、`%H` 24 小时、`%M` 分、`%S` 秒。

### 10.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 不加 `random.seed()` | 每次结果不同，无法核对 | 固定种子 |
| 以为 `randint(1, 10)` 不含 10 | 实际含两端 | 与 `range` 的"含头不含尾"不同 |
| `import datetime` 后写 `datetime.now()` | `AttributeError` | `datetime.datetime.now()` 或 `from datetime import datetime` |
| `strftime` 用了 `%y` | 得到两位年 | 四位年用 `%Y` |
| 用 `random` 做抽奖公平性保证 | 伪随机、可预测 | 安全场景用 `secrets` |

### 10.5.4 随堂示例：random 与 datetime 的常用组合

**核心要点**：验证标准库的两类高频用法：`random` 生成模拟数据（固定 `seed` 便于批改）、`datetime` 做日期推算与格式化，都不需要安装第三方包。

In [ ]:
import random
import datetime

random.seed(20261029)              # 固定种子：每次运行结果一致
roads = ["京港澳", "京开", "机场高速"]
print("随机路段：", random.choice(roads))
print("随机流量：", random.randint(600, 1800))
print("随机车速：", round(random.uniform(15, 85), 1))

d = datetime.date(2026, 11, 12)
print("日期：", d, "| 明天：", d + datetime.timedelta(days=1))
print("格式化：", d.strftime("%Y年%m月%d日"))

**预期输出**：

```text
随机路段： 机场高速
随机流量： 1506
随机车速： 30.8
日期： 2026-11-12 | 明天： 2026-11-13
格式化： 2026年11月12日
```

## 10.6 知识点 10.5　程序包与相对导入

### 10.6.1 核心概念

**程序包**是含 `__init__.py` 的目录，包内可放多个模块：

```text
traffic_pkg/
├── __init__.py      ← 包的入口，定义"对外暴露什么"
├── calc.py          ← 计算模块
└── main.py          ← 包内入口程序
```

`__init__.py` 的三个作用：

1. 标识该目录是包（Python 3.3+ 可省略，但工程上仍建议保留）；
2. 在包被导入时执行一次，常用于"暴露对外接口"；
3. 定义 `__all__`，控制 `from 包 import *` 的行为。

### 10.6.2 语法要点

| 写法 | 含义 |
|---|---|
| `from .calc import capacity` | **相对导入**：从同级的 `calc` 模块导入 |
| `from traffic_pkg import capacity` | 绝对导入：从包顶层导入（需 `__init__.py` 已暴露） |
| `from . import capacity, __version__` | 从包自身（`__init__.py`）导入 |
| `python -m traffic_pkg.main` | 以"模块方式"运行包内入口，相对导入才生效 |

### 10.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 直接 `python traffic_pkg/main.py` | `ImportError: attempted relative import with no known parent package` | 用 `python -m traffic_pkg.main` |
| 在包外目录运行 | `ModuleNotFoundError: traffic_pkg` | 在包的**上一级目录**执行 |
| `__init__.py` 写大量业务逻辑 | 导入即执行，难以测试 | 只放接口与元信息 |
| 忘记 `__all__` | `import *` 暴露内部成员 | 显式声明 |

---

### 10.6.4 随堂示例：包与模块的层级命名

**核心要点**：验证「包是带 `__init__.py` 的目录，模块全名用点号表示层级」；包内 `from . import calc` 中的 `.` 就指当前包，因此包必须用 `python -m` 方式运行。

In [ ]:
import xml.etree.ElementTree as ET      # 标准库中的三层结构：包.子包.模块

print("模块全名：", ET.__name__)
print("所属包：", ET.__package__)
print("名称层级数：", ET.__name__.count(".") + 1)
print("说明：包内 from . import calc 的 . 即当前包")

**预期输出**：

```text
模块全名： xml.etree.ElementTree
所属包： xml.etree
名称层级数： 3
说明：包内 from . import calc 的 . 即当前包
```

# 第二部分　代码示范

> **本部分的阅读与运行方式**：本章代码涉及多个文件，文档中用"**模块文件 `xxx.py`**"标注的代码块，在 Notebook 中对应一个带 `%%writefile` 魔术命令的单元格——运行它等于把这段代码**写盘**为同名文件；随后运行"调用程序"的单元格时，`import` 才能找到该模块。若在终端练习，请照抄代码自行新建同名文件，再执行 `python case1_调用模块.py`。

## 案例 10-1　抽出第一个自定义模块

- **覆盖知识点**：10.1（导入形式）、10.2（模块文档与 `__doc__`）、6.2（默认参数）
- **源文件**：`traffic_utils.py` + `case1_调用模块.py`
- **难度**：基础　**建议用时**：30 分钟

### 任务情境

前几章的"通行能力、车流密度、拥堵判定"在多个程序里反复出现。请把它们抽成独立模块 `traffic_utils.py`，主程序只负责调用。

### 任务要求

1. 新建 `traffic_utils.py`，写入模块级常量 `DEFAULT_CAPACITY` 和三个函数；
2. 模块文件开头写模块文档字符串（三引号），说明用途与作者；
3. 主程序用 `import traffic_utils as tu` 导入，通过 `tu.函数名` 调用；
4. 用 `dir()` 或 `tu.__doc__` 查看模块信息。

### 参考代码

**模块文件 `traffic_utils.py`**

In [ ]:
%%writefile traffic_utils.py
# ---------- 第 10 章 模块与程序包 · 案例 10-1 配套模块：交通计算工具 ----------
# 对应知识点：10.2 自定义模块与模块文档 __doc__、6.2 默认参数、模块级常量
# 功能说明：提供通行能力、车流密度、拥堵判定的通用计算，供其他程序导入复用

"""
交通计算工具模块
作者：智慧交通教研组
说明：提供通行能力、车流密度、拥堵判定的通用计算，供各业务程序复用
"""

DEFAULT_CAPACITY = 2200        # 单车道理论通行能力（辆/h）


def capacity(lanes, per_lane=DEFAULT_CAPACITY):
    """理论通行能力（辆/h）"""
    return lanes * per_lane


def density(cars, length_km, lanes):
    """车流密度（辆/(km·车道)），保留 2 位小数"""
    return round(cars / (length_km * lanes), 2)


def judge(speed, dens):
    """拥堵等级判定"""
    if speed >= 60 and dens < 10:
        return "畅通"
    elif speed >= 40:
        return "缓行"
    elif speed >= 20:
        return "拥堵"
    return "严重拥堵"

**主程序 `case1_调用模块.py`**

In [ ]:
# ---------- 第 10 章 模块与程序包 · 案例 10-1：抽出第一个自定义模块并调用 ----------
# 对应知识点：10.1 import ... as 导入形式、10.2 读取模块文档 __doc__、模块级常量
# 功能说明：以「导入并起别名」的方式调用 traffic_utils 中的函数，演示模块复用

import traffic_utils as tu          # 导入自定义模块并起别名

print("模块说明：", tu.__doc__.strip().splitlines()[0])
print("默认单车道能力：", tu.DEFAULT_CAPACITY)

print("3 车道通行能力：", tu.capacity(3))
print("2 车道(匝道 1800)：", tu.capacity(2, 1800))
print("车流密度：", tu.density(96, 2.5, 3))

speeds = [66, 45, 28, 15]
densities = [8, 12, 22, 45]
for i in range(len(speeds)):
    print("车速", speeds[i], "密度", densities[i], "->", tu.judge(speeds[i], densities[i]))

### 运行结果

```text
模块说明： 交通计算工具模块
默认单车道能力： 2200
3 车道通行能力： 6600
2 车道(匝道 1800)： 3600
车流密度： 12.8
车速 66 密度 8 -> 畅通
车速 45 密度 12 -> 缓行
车速 28 密度 22 -> 拥堵
车速 15 密度 45 -> 严重拥堵
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| 三引号文档字符串置顶 | 10.2 | 模块第一条语句，`__doc__` 才有内容 |
| `DEFAULT_CAPACITY = 2200` | 10.2、6.2 | 模块级常量，作为默认参数的来源（常量只写一次） |
| `def capacity(lanes, per_lane=DEFAULT_CAPACITY)` | 6.2、10.2 | 默认值引用常量，改常量即全局生效 |
| `import traffic_utils as tu` | 10.1 | 别名导入，保留命名空间，读者能看出函数来自哪个模块 |
| `tu.__doc__.strip().splitlines()[0]` | 10.2、4.2 | 取文档首行作为"模块说明" |
| `tu.capacity(3)` / `tu.capacity(2, 1800)` | 10.1、6.2 | 通过模块名访问成员；默认参数可覆盖 |
| `for i in range(len(speeds))` | 10.1、5.2 | 模块里的 `judge()` 被批量调用，验证"写一次、多处用" |

**改一改**：把导入改成 `from traffic_utils import capacity`，再把 `tu.capacity(3)` 改为 `capacity(3)`，其余 `tu.` 前缀全部失效——体会"命名空间"的含义；再修改模块中 `DEFAULT_CAPACITY = 2500`，重跑主程序，观察多处结果同时变化。

## 案例 10-2　用标准库生成模拟车流数据

- **覆盖知识点**：10.4（`random` / `datetime`）、10.1（`import`）、3.2（条件表达式）
- **源文件**：`case2_标准库模拟.py`
- **难度**：基础　**建议用时**：30 分钟

### 任务情境

在没有真实数据的情况下做算法验证，需要生成"看起来像真的"车流数据：随机流量、随机车速、随机路段、真实时间戳。

### 任务要求

1. 用 `random.seed()` 固定随机种子，保证结果可复现；
2. 用 `random.randint()` 生成流量与车速，用 `random.choice()` 抽取路段；
3. 用 `datetime.datetime.now()` 取当前时间并按格式输出；
4. 用 `random.uniform()` 生成浮点车速，`random.random()` 生成 0~1 的拥堵概率。

### 参考代码

In [ ]:
# ---------- 第 10 章 模块与程序包 · 案例 10-2：用标准库生成模拟车流数据 ----------
# 对应知识点：10.4 标准库 random 与 datetime、10.1 import、3.2 条件表达式
# 功能说明：用随机数与时间函数生成带时间戳的模拟车流，并做拥堵概率抽样与日期推算

import random
import datetime

random.seed(20261029)               # 固定种子：每次运行结果一致，便于批改

roads = ["京港澳 K12", "京开 K8", "机场高速 K3", "京承 K20"]

print("=== 模拟车流数据（5 条） ===")
for i in range(5):
    road = random.choice(roads)
    flow = random.randint(600, 1800)          # 整数流量
    speed = round(random.uniform(15, 85), 1)  # 浮点车速
    now = datetime.datetime.now()
    stamp = now.strftime("%Y-%m-%d %H:%M:%S")  # 格式化时间

    print(str(i + 1) + ".", stamp, "|", road, "| 流量", flow, "| 车速", speed)

print()
print("=== 随机拥堵概率 ===")
for i in range(3):
    p = random.random()                        # 0~1 之间
    print("第" + str(i + 1) + "次抽样拥堵概率：", round(p, 3), "->", "拥堵" if p > 0.6 else "不拥堵")

print()
print("=== 时间计算 ===")
today = datetime.date.today()
print("今天：", today)
print("明天：", today + datetime.timedelta(days=1))

### 运行结果（固定种子，随机数值可复现；时间戳为运行时刻）

```text
=== 模拟车流数据（5 条） ===
1. 2026-10-29 11:59:25 | 京承 K20 | 流量 1062 | 车速 59.7
2. 2026-10-29 11:59:25 | 机场高速 K3 | 流量 1542 | 车速 34.8
3. 2026-10-29 11:59:25 | 京承 K20 | 流量 1468 | 车速 76.9
4. 2026-10-29 11:59:25 | 机场高速 K3 | 流量 1135 | 车速 25.3
5. 2026-10-29 11:59:25 | 京开 K8 | 流量 1625 | 车速 24.4

=== 随机拥堵概率 ===
第1次抽样拥堵概率： 0.363 -> 不拥堵
第2次抽样拥堵概率： 0.94 -> 拥堵
第3次抽样拥堵概率： 0.915 -> 拥堵

=== 时间计算 ===
今天： 2026-10-29
明天： 2026-10-30
```

> 若把 `random.seed(20261029)` 换成其他整数，上面 5 条流量与车速会整体变化，但**同一台机器、同一 Python 版本**下数值序列固定，便于课堂统一核对与作业批改。时间戳由 `datetime.now()` 产生，每次运行都不同。

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `import random` / `import datetime` | 10.1、10.4 | 标准库无需安装，导入即用 |
| `random.seed(20261029)` | 10.4 | 固定随机序列，结果可复现 |
| `random.choice(roads)` | 10.4 | 从列表中随机抽一个路段 |
| `random.randint(600, 1800)` | 10.4 | 整数，**含两端** |
| `round(random.uniform(15, 85), 1)` | 10.4、2.6 | 浮点区间抽样后保留 1 位 |
| `datetime.datetime.now()` | 10.4 | `模块.类.方法` 三级访问 |
| `.strftime("%Y-%m-%d %H:%M:%S")` | 10.4、4.5 | 时间格式化，占位符区分大小写 |
| `"拥堵" if p > 0.6 else "不拥堵"` | 3.2 | 条件表达式，省去 `if-else` 块 |
| `today + datetime.timedelta(days=1)` | 10.4 | 日期加减，`timedelta` 表示时间间隔 |

**改一改**：删掉 `random.seed(...)` 连跑两次，对比 5 条数据是否变化；把 `randint(600, 1800)` 改成 `randint(600, 1800) / 100` 观察类型变成 `float`（`randint` 返回整数，除法改变类型）。

---

# 第三部分　布置练习

## 练习 10-1　把统计逻辑模块化

- **对应知识点**：10.2（自定义模块）、10.3（`__main__` 保护）、10.1（两种导入）
- **提交文件**：`flow_stats.py` + `practice1_模块化统计.py`
- **难度**：综合　**建议用时**：30 分钟

### 题目

把"车流量统计"相关逻辑拆成模块 `flow_stats.py`，提供 `total()`、`average()`、`peak_index()` 三个函数；主程序用 `from flow_stats import ...` 导入并调用。

### 要求

1. 模块内三个函数分别返回总量、平均值（保留 1 位）、峰值下标；
2. 模块末尾加 `if __name__ == "__main__":` 自测代码；
3. 主程序用 `from ... import` 导入，并演示一次 `import ... as` 别名调用；
4. 运行主程序时不应触发模块的自测输出。

### 提示

- `if __name__ == "__main__":` 的意义：脚本被直接运行时 `__name__` 为 `"__main__"`，被导入时为模块名，因此自测代码只在直接运行时执行。
- `peak_index` 应返回**下标**而非流量值，方便主程序再算"第几个时段"。
- 两个文件必须放在同一目录；主程序里两种导入方式并存时，注意不要为同一函数起两个名字造成混淆。

### 自检要点（不含答案）

- [ ] 直接运行 `flow_stats.py` 会看到 1 行自测输出；
- [ ] 运行主程序时**没有**自测输出，只有 4 行结果；
- [ ] `total` 返回 8040、`average` 返回 1005.0、`peak_index` 返回 6；
- [ ] 模块文件开头有文档字符串。

## 练习 10-2　搭建一个程序包

- **对应知识点**：10.5（包结构、`__init__.py`、相对导入）
- **提交文件**：`traffic_pkg/__init__.py`、`traffic_pkg/calc.py`、`traffic_pkg/main.py`
- **难度**：进阶　**建议用时**：35 分钟

### 题目

把"计算"与"主程序"分别放进一个程序包 `traffic_pkg` 中：包内 `calc.py` 提供计算函数，`__init__.py` 暴露对外的接口，`main.py` 作为入口。

### 要求

1. 目录结构为 `traffic_pkg/__init__.py`、`traffic_pkg/calc.py`、`traffic_pkg/main.py`；
2. `__init__.py` 中用 `from .calc import ...` 暴露 `capacity` 与 `density`；
3. `main.py` 中用 `from . import capacity` 方式调用；
4. 说明 `__init__.py` 的作用。

### 提示

- 包内模块互相引用要用**相对导入**（`from .calc import capacity`）；`main.py` 需在包的**上一级目录**运行，否则找不到包。
- 运行方式用 `python -m traffic_pkg.main`，而不是 `python traffic_pkg/main.py`。
- `__all__` 与 `__version__` 是包的"门面"，写清便于他人使用；`__init__.py` 内不要放业务逻辑。

### 自检要点（不含答案）

- [ ] 三个文件齐全，目录结构正确；
- [ ] `python -m traffic_pkg.main` 输出 3 行（版本、通行能力、密度）；
- [ ] 直接 `python traffic_pkg/main.py` 会报相对导入错误（能解释原因）；
- [ ] `__init__.py` 中无业务逻辑，只有导入与元信息。

---

# 第四部分　习题答案

## 答案 10-1　把统计逻辑模块化

> 对应 **练习 10-1**（第三部分）

### 参考代码

**模块 `flow_stats.py`**

In [ ]:
%%writefile flow_stats.py
# ---------- 第 10 章 模块与程序包 · 练习 10-1 配套模块：车流量统计 ----------
# 对应知识点：10.2 自定义模块、10.3 if __name__ == "__main__" 保护
# 功能说明：提供总量、均值、峰值下标三个统计函数，并带只在直接运行时才执行的模块自测

"""车流量统计模块"""


def total(flows):
    """车流总量"""
    return sum(flows)


def average(flows):
    """平均流量，保留 1 位小数；空列表返回 0"""
    if len(flows) == 0:
        return 0
    return round(sum(flows) / len(flows), 1)


def peak_index(flows):
    """峰值所在下标（从 0 开始）"""
    return flows.index(max(flows))


if __name__ == "__main__":
    demo = [820, 1100, 760, 690]
    print("[模块自测] 总量", total(demo), "均值", average(demo), "峰值下标", peak_index(demo))

**主程序 `practice1_模块化统计.py`**

In [ ]:
# ---------- 第 10 章 模块与程序包 · 练习 10-1 参考答案：把统计逻辑模块化 ----------
# 对应知识点：10.1 from ... import 与 import ... as 两种导入形式、10.2 自定义模块
# 功能说明：从 flow_stats 导入统计函数，汇总 8 个时段的车流量，并对照别名调用

from flow_stats import total, average, peak_index
import flow_stats as fs

flows = [820, 1100, 760, 690, 720, 980, 1650, 1320]

print("总流量：", total(flows))
print("平均：", average(flows))
print("峰值下标：", peak_index(flows), "（第", peak_index(flows) + 1, "个时段）")

# 别名调用方式，结果一致
print("别名调用总量：", fs.total(flows))

### 运行结果

```text
总流量： 8040
平均： 1005.0
峰值下标： 6 （第 7 个时段）
别名调用总量： 8040
```

### 解析

1. **`__main__` 保护的效果**：主程序导入 `flow_stats` 时 `__name__` 为 `"flow_stats"`，自测块不执行，因此只看到 4 行结果。若把自测代码写在顶层，导入时就会多出一行 `[模块自测]`，污染输出。
2. **两种导入并存**：`from flow_stats import total` 让 `total` 直接可用；`import flow_stats as fs` 保留命名空间。同一模块被导入两次不会重复执行，代价只是命名空间多一个绑定。
3. **`peak_index` 返回下标**：让调用方自己决定展示口径（`+1` 为"第几个"），比返回"第几个时段"更通用。
4. **空列表保护**：`average([])` 会除零，`if len(flows) == 0: return 0` 与第 6 章练习 6-1 的处理一致。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 主程序运行时多一行自测输出 | 自测代码在顶层 | 放进 `__main__` 块 |
| `ModuleNotFoundError` | 两文件不同目录 | 放同一目录 |
| 峰值为 1650 而非 6 | 返回了 `max(flows)` | 返回 `index(max(...))` |
| `NameError: total` | 只写了 `import flow_stats` | 用别名调用 `fs.total` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 模块结构 | 3 | 文档字符串、三个函数 |
| `__main__` 保护 | 3 | 导入时不自测 |
| 两种导入 | 2 | `from` 与 `as` 各一次 |
| 结果正确 | 2 | 8040 / 1005.0 / 6 |

## 答案 10-2　搭建一个程序包

> 对应 **练习 10-2**（第三部分）

### 参考代码

**`traffic_pkg/__init__.py`**

In [ ]:
%%writefile traffic_pkg/__init__.py
# ---------- 第 10 章 模块与程序包 · 练习 10-2 配套程序包：包声明 ----------
# 对应知识点：10.5 包结构与 __init__.py、__all__ 导出、包版本号
# 功能说明：声明 traffic_pkg 包，把 calc 中的接口对外暴露，并给出 __all__ 与 __version__

"""智慧交通计算包：对外暴露常用计算接口"""

from .calc import capacity, density

__all__ = ["capacity", "density"]
__version__ = "1.0.0"

**`traffic_pkg/calc.py`**

In [ ]:
%%writefile traffic_pkg/calc.py
# ---------- 第 10 章 模块与程序包 · 练习 10-2 配套程序包：包内计算模块 ----------
# 对应知识点：10.5 程序包内的子模块
# 功能说明：提供通行能力与车流密度两个计算函数，供包入口与其他模块调用

"""包内计算模块"""


def capacity(lanes, per_lane=2200):
    """理论通行能力（辆/h）"""
    return lanes * per_lane


def density(cars, length_km, lanes):
    """车流密度（辆/(km·车道)），保留 2 位小数"""
    return round(cars / (length_km * lanes), 2)

**`traffic_pkg/main.py`**

In [ ]:
%%writefile traffic_pkg/main.py
# ---------- 第 10 章 模块与程序包 · 练习 10-2 配套程序包：包入口 ----------
# 对应知识点：10.5 相对导入、10.3 __main__ 保护；运行方式 python -m traffic_pkg.main
# 功能说明：以模块方式运行包入口，打印包版本并调用包内计算接口

"""包入口程序：需在 traffic_pkg 的上一级目录执行 python -m traffic_pkg.main"""

from . import capacity, density, __version__

if __name__ == "__main__":
    print("包版本：", __version__)
    print("3 车道通行能力：", capacity(3))
    print("车流密度：", density(96, 2.5, 3))

运行方式与结果：

```text
$ python -m traffic_pkg.main
包版本： 1.0.0
3 车道通行能力： 6600
车流密度： 12.8
```

### 解析

1. **`__init__.py` 的作用**：包被导入时执行一次，把 `calc` 里的函数"提到"包这一层，于是外部可以写 `from traffic_pkg import capacity` 而不必知道 `calc` 的存在。这是包的"门面"设计。
2. **相对导入 `.`**：`from .calc import ...` 中的 `.` 指"当前包"。它只在"以模块方式运行"（`-m`）或有包上下文时有效，因此 `python traffic_pkg/main.py` 会报 `attempted relative import with no known parent package`。
3. **`__all__` 的作用**：限制 `from traffic_pkg import *` 导出的名称，避免把 `calc`、`__version__` 等内部成员一并暴露。
4. **版本号的意义**：`__version__` 是包的元信息，第 13 章发布与排查问题时都要用到。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `ImportError: attempted relative import` | 直接运行 `main.py` | `python -m traffic_pkg.main` |
| `ModuleNotFoundError: traffic_pkg` | 在包内目录运行 | 回到上一级目录 |
| `ImportError: cannot import name 'capacity'` | `__init__.py` 未暴露 | 加 `from .calc import capacity` |
| 包内出现业务逻辑 | `__init__.py` 里写计算 | 移到 `calc.py` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 目录与文件 | 3 | 三文件齐全、命名正确 |
| `__init__.py` 暴露接口 | 3 | 相对导入 + `__all__` |
| 入口运行 | 2 | `-m` 方式输出 3 行 |
| 结构说明 | 2 | 能解释 `__init__.py` 与相对导入 |

---

## 本章小结

- **模块**：一个 `.py` 文件；三种导入形式按"是否需要命名空间"选择，`import ... as` 最常用。
- **自定义模块**：文档字符串置顶、常量在上、函数在下、末尾用 `__main__` 保护自测。
- **`__name__`**：直接运行是 `"__main__"`，被导入是模块名。
- **标准库**：`random` 固定种子便于复现；`datetime` 用 `strftime` 格式化、`timedelta` 做日期加减。
- **程序包**：目录 + `__init__.py`；包内用相对导入，入口用 `python -m 包.模块` 运行。

## 自检清单

- [ ] 能说出四种导入写法的区别与适用场景；
- [ ] 知道模块文档字符串为什么必须在第一行；
- [ ] 能解释 `if __name__ == "__main__":` 的两侧行为；
- [ ] 会用 `seed` / `randint` / `uniform` / `choice` / `random`；
- [ ] 会用 `strftime` 与 `timedelta`；
- [ ] 能搭出"包 + `__init__.py` + 相对导入"的最小结构，并说明 `-m` 运行的必要性。

## 前后章节衔接

- 本章的模块拆分让**第 11、12 章**的类可以独立成文件，第 13 章的综合项目正是"多模块 + 主程序"的组织方式。
- 本章的 `random` 将在**第 13 章**用于车流仿真的随机到达。
- 本章的"接口暴露"思想（`__init__.py` 的 `__all__`）与**第 12 章**的"基类约定接口"一脉相承：都是"对外承诺一组能力"。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：异常处理](../09异常处理/09_异常处理.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：面向对象编程上 ▶](../11面向对象编程上/11_面向对象编程上.ipynb)